# Dynamic Model Generation

Don't want to define a model manually? Let the LLM create one from a natural language description.

The LLM intelligently decides:
- Which fields should use statistical distributions
- Which fields should be LLM-generated
- Appropriate distribution parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from gendantic import (
    generate_model_from_description,
    generate_synthetic_data,
)

plt.style.use('seaborn-v0_8-whitegrid')

## Basic Model Generation

Describe what you want, and the LLM creates a Pydantic model:

In [ ]:
description = """
A customer support ticket with:
- Priority level (high, medium, low)
- Category (billing, technical, general inquiry)
- Customer satisfaction score (0-10)
- Resolution time in hours
- A description of the issue
"""

TicketModel, source_code = await generate_model_from_description(
    description, 
    model_name="SupportTicket"
)

print("Generated model code:")
print("-" * 50)
print(source_code)
print("-" * 50)

In [ ]:
# Generate data using the model
tickets = await generate_synthetic_data(TicketModel, count=5, seed=42)

print("Generated tickets:")
for i, ticket in enumerate(tickets, 1):
    print(f"\nTicket {i}:")
    for field_name in TicketModel.model_fields:
        value = getattr(ticket, field_name)
        if isinstance(value, float):
            print(f"  {field_name}: {value:.2f}")
        elif isinstance(value, str) and len(value) > 60:
            print(f"  {field_name}: {value[:60]}...")
        else:
            print(f"  {field_name}: {value}")

## Complex Business Models

Generate more complex domain-specific models:

In [ ]:
ecommerce_description = """
An e-commerce order with:
- Customer name and email
- Order total (typically £20-500 with some larger orders)
- Number of items (usually 1-5)
- Shipping method (standard 60%, express 30%, overnight 10%)
- Discount percentage applied (most orders have small or no discount)
- Order status (pending, processing, shipped, delivered)
- Brief order notes
"""

OrderModel, order_code = await generate_model_from_description(
    ecommerce_description,
    model_name="EcommerceOrder"
)

print("Generated Order Model:")
print(order_code)

In [ ]:
orders = await generate_synthetic_data(OrderModel, count=100, seed=42)

# Visualize the generated distributions
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

# Order total distribution
totals = [o.order_total for o in orders]
axes[0].hist(totals, bins=20, edgecolor='white', alpha=0.7)
axes[0].set_xlabel('Order Total (£)')
axes[0].set_ylabel('Count')
axes[0].set_title('Order Total Distribution')

# Shipping method
shipping = Counter(o.shipping_method for o in orders)
axes[1].bar(shipping.keys(), shipping.values(), edgecolor='white', alpha=0.7, color='green')
axes[1].set_xlabel('Shipping Method')
axes[1].set_ylabel('Count')
axes[1].set_title('Shipping Method Distribution')

# Items vs Total (correlation)
items = [o.number_of_items for o in orders]
axes[2].scatter(items, totals, alpha=0.5, s=30, color='orange')
axes[2].set_xlabel('Number of Items')
axes[2].set_ylabel('Order Total (£)')
axes[2].set_title(f'Items vs Total (r={np.corrcoef(items, totals)[0,1]:.2f})')

plt.tight_layout()
plt.show()

print("\nSample orders:")
for order in orders[:3]:
    print(f"  {order.customer_name}: {order.number_of_items} items, £{order.order_total:.2f}, {order.shipping_method}")

## Healthcare Data Model

In [ ]:
patient_description = """
A patient medical record with:
- patient_name and date of birth
- Blood pressure (systolic and diastolic)
- Heart rate (beats per minute)
- BMI
- Cholesterol level
- Risk category (low, moderate, high)
- Primary care notes
"""

PatientModel, patient_code = await generate_model_from_description(
    patient_description,
    model_name="PatientRecord"
)

print(patient_code)

In [ ]:
patients = await generate_synthetic_data(PatientModel, count=3)

for patient in patients:
    print(f"\n{patient.patient_name}")
    for field in PatientModel.model_fields:
        if field != "patient_name":
            val = getattr(patient, field)
            if isinstance(val, float):
                print(f"  {field}: {val:.1f}")
            else:
                print(f"  {field}: {val}")

## Financial Data Model

In [ ]:
transaction_description = """
A financial transaction with:
- Account holder name
- Transaction amount (typically small purchases but occasionally large)
- Transaction type (purchase 70%, transfer 20%, withdrawal 10%)
- Merchant category (retail, food, travel, utilities, entertainment)
- Risk score (0-1, most transactions are low risk)
- Timestamp
- Location
"""

TransactionModel, tx_code = await generate_model_from_description(
    transaction_description,
    model_name="Transaction"
)

print(tx_code)

In [ ]:
transactions = await generate_synthetic_data(TransactionModel, count=5, seed=42)

for tx in transactions:
    print(f"\n{tx.account_holder_name}")
    for field in TransactionModel.model_fields:
        if field != "account_holder_name":
            val = getattr(tx, field)
            if isinstance(val, float) and "amount" in field.lower():
                print(f"  {field}: £{val:,.2f}")
            elif isinstance(val, float):
                print(f"  {field}: {val:.3f}")
            else:
                print(f"  {field}: {val}")

## Model with Correlations

The LLM can also suggest correlations when you describe relationships:

In [ ]:
employee_description = """
An employee record with:
- Name and email
- Age (22-65)
- Years of experience (correlated with age - older employees have more experience)
- Salary (correlated with experience - more experience means higher salary)
- Performance score (0-1)
- Department (Engineering 40%, Product 25%, Sales 20%, Other 15%)
- Job title

Note: Age, experience, and salary should be positively correlated.
"""

EmployeeModel, emp_code = await generate_model_from_description(
    employee_description,
    model_name="Employee"
)

print(emp_code)

In [ ]:
employees = await generate_synthetic_data(EmployeeModel, count=100, seed=42)

# Check if correlations were applied
ages = [e.age for e in employees]
exp = [e.years_experience for e in employees]
sal = [e.salary for e in employees]

print("Correlation analysis:")
print(f"  Age-Experience: {np.corrcoef(ages, exp)[0,1]:.2f}")
print(f"  Experience-Salary: {np.corrcoef(exp, sal)[0,1]:.2f}")

# Visualize correlations
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(ages, exp, alpha=0.5, s=30)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Years Experience')
axes[0].set_title(f'Age vs Experience (r={np.corrcoef(ages, exp)[0,1]:.2f})')

axes[1].scatter(exp, sal, alpha=0.5, s=30, color='green')
axes[1].set_xlabel('Years Experience')
axes[1].set_ylabel('Salary (£)')
axes[1].set_title(f'Experience vs Salary (r={np.corrcoef(exp, sal)[0,1]:.2f})')

plt.tight_layout()
plt.show()

print("\nSample employees:")
for emp in employees[:5]:
    print(f"  {emp.name}: Age {emp.age}, {emp.years_experience} yrs exp, £{emp.salary:,.0f}")

## Security

Generated code is validated via AST parsing before execution. The following are blocked:
- Import statements
- `exec`, `eval`, `compile`, `open`
- `globals()`, `locals()`, `__builtins__`
- Dunder attribute access (`__class__`, `__bases__`, etc.)

## Tips for Good Descriptions

1. **Be specific about ranges**: "Age (22-65)" is better than "age"
2. **Specify distributions for categories**: "Status (pending 40%, shipped 35%, delivered 25%)"
3. **Mention correlations**: "Experience correlates with salary"
4. **Describe semantic fields**: "Brief professional bio" helps the LLM generate better text
5. **Use domain terminology**: The LLM understands business, medical, financial terms